# <div align="center">🪐 Madar (مدار) - 14B Frontier AI on Google Colab</div>
### <div align="center">Frontier-Class 14B Parameter AI Inference with NVIDIA Tesla T4 OpenCL Zero-Copy</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![Parameters](https://img.shields.io/badge/Model%20Scale-14%20Billion-red.svg)]() 
[![VRAM](https://img.shields.io/badge/GPU%20VRAM-15%20GB%20Tesla%20T4-green.svg)]() 
[![Acceleration](https://img.shields.io/badge/Zero--Copy-OpenCL%20mmap-purple.svg)]()

---

Welcome to the **Madar 14B Frontier Edition** notebook.

This notebook pushes Google Colab's free GPU tier (NVIDIA Tesla T4 with 15.36 GB VRAM) to its peak performance using zero-copy memory mapping and OpenCL hardware acceleration:

### 🌟 Why 14B Models are a Massive Leap Forward:
1. **Scale (14.7 Billion Parameters):** 48 full transformer layers, 5,120 hidden dimension, and 40 multi-head attention heads.
2. **Perfect Fit for Tesla T4 (15 GB VRAM):** The Q4_0 quantized model is **~8.54 GB**, utilizing only ~56% of the 15.36 GB VRAM, leaving over 6 GB of VRAM for deep KV-cache and expanded context windows!
3. **State-of-the-Art Intelligence:** Models like **Qwen2.5-14B** and **DeepSeek-R1-Distill-14B** rival proprietary models on complex math, advanced reasoning, coding, and multilingual fluency.
4. **Zero-Copy Instant Mounting:** Zero-copy `mmap` mounts the 8.5 GB weights instantaneously without duplicate RAM buffering.

## ⚙️ Step 1: Check GPU VRAM & Hardware Features
Verify that the attached Tesla T4 GPU has ~15 GB VRAM available:

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 Step 2: Build Madar Core & AlQalam C++20 Extension

In [ ]:
import os, sys

if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull

!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev curl libcurl4-openssl-dev poppler-utils
!bash tools/build_linux.sh
!bin/ring -v

## 🪄 Step 3: Register `%%madar` Interactive Cell Magic

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic to execute Madar code directly."""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([MADAR_BIN, tmp_file], capture_output=True, text=True, env=env, cwd=cwd)
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print(res.stderr, end="", file=sys.stderr)
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ %%madar cell magic registered successfully!")

## 🧠 Step 4: Download 14B Frontier Models (1-Click Downloads)
Colab's download speed reaches 80-120 MB/s, taking ~80-100 seconds to download the complete 8.54 GB model:

In [ ]:
# 🥇 [Recommended] Download Qwen2.5-14B-Instruct (~8.54 GB)
# Exceptional multilingual, math, reasoning, and coding capabilities
!bash tools/download_colab_models.sh qwen14b

In [ ]:
# 🥈 Download DeepSeek-R1-Distill-Qwen-14B (~8.54 GB)
# Superhuman reasoning with transparent Chain-of-Thought (CoT)
!bash tools/download_colab_models.sh deepseek14b

## ⚡ Step 5: Madar 14B Frontier Benchmark Suite

### 🚀 Benchmark 5.1: Multi-Step Physics & Kinematics Problem Solving

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

see f"[MADAR 14B] Mounting model: {cModelPath}" + nl
t0 = clock()
oModel = ai_open(cModelPath)
t1 = clock()
see f"[MADAR 14B] Mounted in: {(t1 - t0) / clockspersecond()} sec via Zero-Copy mmap!" + nl

oModel.setThreads(4)
oModel.setGPU(1)
oModel.setTemperature(0.5)

cPrompt = "Solve step-by-step with clear formulas: A car starts from rest with a constant acceleration of 2.5 m/s² for 12 seconds, then cruises at constant speed for 45 seconds, and finally brakes to a complete stop with a deceleration of 3.0 m/s². Calculate: 1) Maximum speed achieved in km/h. 2) Total distance traveled in meters. 3) Total duration of the trip in seconds."

see nl + "========================================================" + nl
see "Problem: " + cPrompt + nl
see "========================================================" + nl

tGen0 = clock()
cAnswer = oModel.generate(cPrompt, 768)
tGen1 = clock()

see oModel.extractAnswer(cAnswer) + nl
see "========================================================" + nl
see f"Speed: {oModel.tokensPerSecond()} tokens/sec (Latency: {(tGen1 - tGen0)/clockspersecond()} sec)" + nl

### 🧠 Benchmark 5.2: Complex Deductive Logic & Chain-of-Thought (CoT)

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)
oModel.setTemperature(0.6)

cLuzzle = "Three wise men are shown 5 hats: 3 white and 2 black. Blindfolded, one hat is placed on each man's head, and the other 2 are hidden. When unblindfolded, each can see the hats of the other two, but not their own. The first wise man is asked if he knows his hat color; he replies: 'No'. The second is asked; he also replies: 'No'. The third wise man, who is completely blind, is asked; he immediately replies: 'Yes, I know my color!' What is the color of the blind man's hat, and what was his logical deduction?"

see "========================================================" + nl
see "Logic Puzzle: " + cLuzzle + nl
see "========================================================" + nl

t0 = clock()
cFull = oModel.generate(cLuzzle, 1024)
t1 = clock()

cThinking = oModel.extractThinking(cFull)
cAnswer   = oModel.extractAnswer(cFull)

if cThinking != ""
    see "--- [Internal Chain of Thought] ---" + nl
    see cThinking + nl
    see "-----------------------------------" + nl
ok

see nl + "--- [Final Solution] ---" + nl
see cAnswer + nl
see "========================================================" + nl
see f"Speed: {oModel.tokensPerSecond()} tokens/sec (Time: {(t1 - t0)/clockspersecond()} sec)" + nl

### 💻 Benchmark 5.3: Full Data Structure Implementation in Ring / Madar

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)
oModel.setTemperature(0.2)

cTask = "Write an object-oriented Ring / Madar implementation of a Binary Search Tree (BST) featuring:\n1. Class Node and class BST.\n2. insert(value) method.\n3. search(value) method returning boolean.\n4. inorder() method displaying sorted values.\nProvide runnable demonstration code with explanatory comments."

see "========================================================" + nl
see "Task: " + cTask + nl
see "========================================================" + nl

see oModel.generate(cTask, 700) + nl

### 🌊 Benchmark 5.4: Real-Time 14B Token Streaming

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oModel = ai_open(cModelPath)
oModel.setGPU(1)
oModel.setThreads(4)

cPrompt = "Write an insightful essay discussing how native programming languages like Madar fuse zero-overhead C systems performance with local AI generative workflows."

see "================ [Real-Time 14B Streaming] ================" + nl
oModel.generateStream(cPrompt, 400)
see nl + "===========================================================" + nl
see f"Streaming Speed: {oModel.tokensPerSecond()} tokens/sec" + nl

## ✒️ Step 6: AlQalam C++20 Hybrid RAG with 14B Brain

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/madar_ai.ring"
load "libraries/madar_ai/rag.ring"
load "alqalam.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oTimer = new QalamChronos()
oTimer.reset()

see "[RAG 14B] Initializing 14B model for RAG... " + nl
llm = ai_open(cModelPath)
llm.setGPU(1)
llm.setThreads(4)
store = new VectorStore(llm)

store.addText("Madar combines native AVX2 SIMD with OpenCL zero-copy GPU acceleration to infer 14B parameter models on consumer hardware.")
store.addText("The AlQalam C++20 library provides high-speed QalamIndex inverted indexing blended with dense cosine similarity.")
store.addText("Quantum supremacy involves performing calculations impossible for traditional supercomputers using superposition.")

cQuestion = "How does the Madar runtime enable 14B parameter models to run efficiently on GPUs?"
see nl + "Question: " + cQuestion + nl

oTimer.reset()
aResults = store.hybridSearch(cQuestion, 2, 0.7)
see f"[RAG] AlQalam Hybrid Search Results (Time: {oTimer.elapsed()}):" + nl

for res in aResults
    see f"  - [Score: {res[1]}] (Semantic: {res[3]} | Lexical: {res[4]}) {res[2].cText}" + nl
next

best_text = aResults[1][2].cText
see nl + "[RAG 14B] Grounded Response:" + nl
cPrompt = f"Answer precisely and comprehensively based on this context: {best_text}\nQuestion: {cQuestion}"
see llm.generate(cPrompt, 350) + nl

## 🤖 Step 7: Autonomous AIAgent with 14B Intelligence

In [ ]:
%%madar
load "stdlibcore.ring"
load "libraries/madar_ai/agent.ring"

cModelPath = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModelPath) cModelPath = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok

oAgentModel = ai_open(cModelPath)
oAgentModel.setGPU(1)
oAgentModel.setThreads(4)
oAgent = new AIAgent(oAgentModel)
oAgent.remember("Madar is an AI-native runtime for the Ring ecosystem featuring zero-copy GPU inference.")
oAgent.remember("Madar supports 14B models on Colab via AVX2 and OpenCL zero-copy acceleration.")

see "[AGENT 14B] Vector memory ready! Querying autonomous agent:" + nl

cQuery = "Calculate 8945 multiplied by 762 using the calc tool, and summarize what Madar is from your memory."
see nl + "User > " + cQuery + nl + nl

cResponse = oAgent.query(cQuery)
see nl + "Agent (14B Brain) > " + nl + cResponse + nl

## 💬 Step 8: Interactive In-Notebook Chat with 14B Model

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import subprocess, os, tempfile

chat_history = []

out_area = widgets.Output()
text_input = widgets.Text(
    placeholder='Ask the 14B model any challenging question...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(
    description='Send 🚀',
    button_style='success',
    layout=widgets.Layout(width='20%')
)

def ask_madar_14b(prompt):
    ring_code = f'''
load "libraries/madar_ai/madar_ai.ring"
cModel = "models/qwen2.5-14b-instruct-q4_0.gguf"
if not fexists(cModel) cModel = "models/deepseek-r1-distill-qwen-14b-q4_0.gguf" ok
model = ai_open(cModel)
model.setGPU(1)
model.setThreads(4)
model.setTemperature(0.6)
model.setMaxTokens(350)
see model.generate("{prompt}", 350)
'''
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(ring_code)
        tmp_name = f.name
    
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = os.path.abspath("lib") + ":" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([os.path.abspath("bin/ring"), tmp_name], capture_output=True, text=True, env=env, cwd=cwd)
        return res.stdout.strip()
    finally:
        if os.path.exists(tmp_name):
            os.remove(tmp_name)

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    
    with out_area:
        display(HTML(f'<div style="background-color:#e8f5e9; padding:12px; border-radius:10px; margin:6px 0;"><b>🧑 You:</b> {user_msg}</div>'))
        display(HTML('<div id="loading" style="color:#2e7d32;"><i>⏳ 14B Frontier brain is reasoning...</i></div>'))
    
    bot_msg = ask_madar_14b(user_msg)
    
    with out_area:
        clear_output(wait=True)
        for role, text in chat_history:
            bg = '#e8f5e9' if role == 'user' else '#f3e5f5'
            author = '🧑 You' if role == 'user' else '🪐 Madar (14B)'
            display(HTML(f'<div style="background-color:{bg}; padding:12px; border-radius:10px; margin:6px 0;"><b>{author}:</b> {text}</div>'))
        
        display(HTML(f'<div style="background-color:#f3e5f5; padding:12px; border-radius:10px; margin:6px 0;"><b>🪐 Madar (14B):</b> {bot_msg}</div>'))
    
    chat_history.append(('user', user_msg))
    chat_history.append(('bot', bot_msg))

send_btn.on_click(on_send)
text_input.on_submit(on_send)

controls = widgets.HBox([text_input, send_btn])
display(widgets.VBox([out_area, controls]))